# A pre-norm transformer block from scratch

Build one pre-norm transformer encoder block without PyTorch's attention or transformer modules, and count its parameters by formula, as Round 2 asked in 2026 (Problem 1 banned `nn.MultiheadAttention` and `nn.TransformerEncoder`).

```python
class PreNormBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        # Your code
    def forward(self, x, causal=False):
        # Your code

def block_param_count(d_model, n_heads, d_ff):
    # Your code: return an int
```

**Layers.** Create exactly these attributes (all `nn.Linear` layers with biases):

| attribute | layer |
|---|---|
| `self.ln1`, `self.ln2` | `nn.LayerNorm(d_model)` |
| `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o` | `nn.Linear(d_model, d_model)` |
| `self.fc1` | `nn.Linear(d_model, d_ff)` |
| `self.fc2` | `nn.Linear(d_ff, d_model)` |

**Forward.** `x` has shape `(B, T, d_model)`; `d_model` is divisible by `n_heads` and each head has size `d_head = d_model // n_heads`, using features `h * d_head` to `(h + 1) * d_head - 1` of the projections.

$$h = x + W_o\big(\operatorname{MHA}(\operatorname{ln1}(x))\big), \qquad y = h + \operatorname{fc2}\big(\operatorname{GELU}(\operatorname{fc1}(\operatorname{ln2}(h)))\big),$$

where MHA is multi-head scaled dot-product self-attention (scale $1/\sqrt{d_{\text{head}}}$) whose head outputs are concatenated in head order before `W_o`, and GELU is the exact one (`torch.nn.functional.gelu` with its default). If `causal=True`, query $i$ attends only to keys $j \le i$. No dropout. Return `y`, shape `(B, T, d_model)`.

**Count.** `block_param_count(d_model, n_heads, d_ff)` returns the number of parameters of this block as a Python `int`, computed from the formula (it must work instantly for `d_model = 8192`, so do not build the module).

**Example.**

```python
torch.manual_seed(0)
blk = PreNormBlock(32, 4, 128)
blk(torch.randn(2, 6, 32), causal=True).shape       # torch.Size([2, 6, 32])
block_param_count(32, 4, 128)                       # 12704
```

**Restrictions.** Only `import torch`, `import torch.nn as nn`, `import torch.nn.functional as F` and `import math`. Banned: `nn.MultiheadAttention`, `nn.TransformerEncoderLayer`, `nn.TransformerEncoder`, `nn.TransformerDecoderLayer`, `nn.Transformer`, `F.scaled_dot_product_attention`, `multi_head_attention_forward`, and `numel`.

**Constraints.** $d_{\text{model}} \le 256$ and $T \le 256$ for the module; the count must handle $d_{\text{model}}, d_{\text{ff}} \le 10^5$. Time limit 1 second per forward call on CPU.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem1_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem1_submission.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class PreNormBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        # Your code

    def forward(self, x, causal=False):
        # Your code
        return x


def block_param_count(d_model, n_heads, d_ff):
    # Your code
    return 0


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem1_submission as submission
importlib.reload(submission)
# e.g. submission.PreNormBlock(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9Wwtz20aS/itzTNUKcCiYpJ9LR76TEyXxlV9la2v3SlIhQ2JIIQIHCAa0pNXqv9/XPQNgAJKy5eydU6GAeXT39PR7BjeDNBlMxWA9frpfjMaDoRisVCXRdLPZU6VVpqjtUBSl2td5uRJVKbVZ4EmVYpbl8wuxKPOVMPNSVvNzmpUpY3Jdg8ocklSvK2XQOhnhNUkXi3S+zqprtDxGg9JVSc+DD6V6B+ivCDRPzJN1pmItV0xJUeazTK3GsVnPVqkxKTBhlMyy/FIlcboq8rIiNCeDKi8tQStZnQ/O8DCTWquEO98Cd3quZHJYVcDtwBy3izvS8zxR5Rt5rcrtXb3Wn9TOCfRq5jIDgUlexVhDsp5XsfRRr4igmChq22NMv5RlQv16vVJZvYo4y/OCVrmQmVG0U8pUcZWuVL6u4obLlVw6VrS0RLOas34jtpd2d6NZluB7pcp4nq91xd3E1cjuCpNj8nU5573Jy3SZapnRMMysJmisyrW6xfsizXj3bwbnstSQkKjg7T7l/35VWaFKI6pzWQm3xWJZyoQa7Z6K4DcWNDe9bn3wWxid6lN96IaL1Ih5romBD9c6rczDH+jPy4cOKt6LNHn50I4GEZF4XYlELVJAFcRGsVjrOXHfCBK6hBvjB5a2Sl4oPKlT3cqfsMwQv6Hpt0gc/ePw7Yc3R59ElpqKlqQsWAJmhDnPL7XApI9rLYLPqUlBVfhCfCJwlSjXQKs+q/KaJ2FpP8s0W5eYWcrUKEGv4jKtzoUUgGfkUonLMiWJEdg08UrpqTDyWlwSuZfSoDfXy6HQBFUk61UhztMkwfBUF+vKEPfsLmjH0nleXDcvJFPEX/cKKSyuBaDqglq/Ez/mmZyJHz/8zQhZKmFIDUvilSZS9ox4OxEfynwoTC4geDIjiKleGrFagykzRUwFi3KyBym6s5T4MMP6z7GVCzmHwEWEaRKNaHOlWK5p/wOL2QBWpvar8xKaI0yhsGEYVebr5Xl2DWnJFgK0gIp5XhKjS0VMr0S6IALxygbDrcNpEDHl04ejo5/inw9/PH7/URwQdlrwqZ5nEthpG4Kjq7kqSFLC6akW+Fegy46CQIHjKgkgi8lQrMyyHgO8Oicm68S10L92cwMa2wKJdRFc1XNhIr059Q6RQnqtC6w/1aaSeq6Cq6Htj46VNnkZetMZrarWpRZXUQIfMD8PwmherPHL2xyEdrDiZYrXjO6oLPPSg2KX7MHSRSSNLEt5DbrbdZhzWahgmVdDCKWmX8jnwekAkg3DdzqoKTPgdbUuMhXQyjE+jHiqo4V5ijH1IIIVDsXidHBDAG+x4cbiEjcGdkddFWpeQSZuvPG3QNdSNs9ycxdlQ1FWeXYwVvtPh0K6x+c1vdCsS5DcECtNdY2FLrJcEmHUzji7Hd5ilnZ9tKRL+7hjOXYBbni4bW11Fy+vQYANSQ3MW1phNujIsiAUsBQkhX7fpevz0ZMllSlbQk1zUr1ogDtBpv3OMsdEMMOxi34cu+jHFzxVluAYM4KIW8kr+iNnJliKfXEZhiHBxkLTfyqh4N/EiJRvi6p0+cSGTnyWGeyDCDJZLsnsKhJYcYM/0+jR8jbs7L36Yy2zr5DKrlLZoUS0TljShyFzFETAdZeu/3TA29EC2SIunoC0Y/7PxcIXDaY/dnwAcRvK5HO1gUC7Mu1TDGEgannxLQwY4xt0/UfZIY1GoamzGamJWUV4O+jJ7NwPRujtSDtlg35qBSsc4DCKOZSM4w45S5g/2jmHda8etFdD7VIaGy0LePLq/9c0Z7lW9zTKQFlL5RWJJPv2QQiXmLD2dukJknQORlLgEvr0dBfXoY4ABolSxcExIj1PwByBx2Bfn7xNAGHHhVBTRDC576rDeGxLIIdi9u18l1/FdzvE6gXj+1a2S2I7wzFdU+AQSYvEBLNwm63B9NbObJ3e2ImZe+S97Wo2rWBorR0kW/t7VcNhCF7wkserNZQC0U2w0EPxAOYUmvHgwcUlPdWU2NDxR3gAsdCMeUEBKq0BMQ6iwCXUS2qEjOUSCYyGxGlRZHKuojrqZK9P+gS7eNJqlgw5npU0gTCe2ZGIzsh56KBPkO2mKSkEGQs2IU1VlDchNVXBP9MisHMYR0fA2Z40wuWMyDUSG0E5HQX46SKlyJJC5nYt392k4nsxvn0hLvPygqJ6ycIraPsQjzb20jEZxLccpiA+1YFRFAqCqq/l8gufxbCufsi9JZLmzBj+0OafzCCOdfeamQjebQ5FqN4DREkpGBKklaElcRz+Vs7FHAhoChZpo27KVjiwp+AA2cX+bJ3AklIzx+AIy1NKoNYF1ElUl+lcEUBwKIWHzhdOg0FzzVHDIkQtC8RL5L4JguDk00TimDrSEs1YTLZHKMgvWGTEYZV0xKoaUTQJCBGyy4XNYVVZW5sviFJSbZ+MGKVy4Yhb74FwmygeCj9taEUyJlEsSRuCiS932ESg+eHAQeqZohlymou2SU6G4mICbF37yCQPe42dlXyZFUwnMWLCbJh4zcwGZGxBQi52Bzv8eNNfEOtRlecsN0j+k2oaTRa3iPIhZeqFy7oaIWZhgtoxILHWlM/fONa4eZxDIrGrt3jLP0aJzBJp8AKitaQA1iaZN26fGBBpx50KWuoltFMl9Ya1CQ52MslX4PZCQvziZuApV3LWRso0j7nC0Kly/EIt0ABQ1hYPUjYa52mWkNbNqTKChDklXaCR//3p/TvQqllzTJWAQtbV4ro6R8v+a1HjET+4EkecpOVL8UOLA514h1b+i9uqlxspP5TNS/+xsVdVls6aJvsHLdG6SrO2OW8efzd5WyQoZHXuTzfpUst2mrk2vfJC/VLCL8zk/KJpgRRrKhbYWoOZp1imVo25Q8NFWu1nCsPEOHoajV1MZ6ZUJ1pRJXLIJgXZhXM+tfErZAouL091jSNapBkkun4NTgcgOy8VZYCu0oJINHrwN32hqYZj8owqKjlXAIAwLWjXogcuUvz16OMRNMfxIvqAv0EcUxksjsMIhoumI6orZAnKTjW4EtHgCNKqyioYQRkQPhCYEAC31b64FiS+g6/+Q07F0ePRhCthb/5++D+f4leH794d/QQKbk4HsauLxjEtBgur99O+qis1d0+I8+3TPF8VoPV0cHuq37x//yF+9/6no08AF0BWop9z6Co9/B2Cq+zjobnW86bjDWLIHwHDvn1S3stPCDLbt1+UJv+cl0dXReiXWI5tLeZLVZZYItNbBSRj6xXsDRUsG43lVLEG5FcjKLKZx/NzNb8wgSnntMWV7Prbj1bfKRqGsyI+5heWOSuzBGNewCxSBGhrfKw2ptprdoeDIVdKJEdVlaktK3ZclHVCJ2fbotmqVAjsmEuQEuTXILQbh366Rn5+xXEoFeXUZnx44sieQlq4VOyon5KZNDzdpci5tnbmRkX0V+e3U3rGaCQ/t45AV2cnlwB+RTDM0JNe8Z14dHLmCJ3Jpp39ZFW7HC6HxmvDwLz2GfQhtrVS104ijpARupsmVBdHJFbaQMECBgQ7PqCMhoLudEZnDVjOBcduKotIsB0CLp17PF/JarXO/JYFV0dgqsk2g/mXMrsIaDN6rtsL02m4lWebGvSziiaUpYERkzvddGCupIJ4OZA8iILyFZUW7GsEbGAkRY7BCpEnzBElcwR8xcTajQin232jtxeRTBKHI9wc3O6NNy4yRZbSdhPOk/3x2ZaJ7eZF6yKhqHvLXD9JzO5m48+wev3VYEOxV8xHV3jH8luF2sJLnuEYiec/wcU+BwGtxwWPdR0ObJOA3tRdzPumqbzO+zL9sNad/tp7AsF0kKJ9FdR3nkW+C2Ca7F7V9kHbMLbOqo+TFT+SRaF0EtQVHzvpdGBNnjXv/9nl1zYs7AVfpfq982KH6+WhIR8U2tpKb0KUu3FvZUUnjxsybW3QPYkDYZ5AeiApSHSgdpr/OgqwWmJFnlpZIWhc9HtO6Skb2MDDE4bbrcv35FHAmBrUzd5Q7FkgtUaRAu4BH/Le5R6VYm+31xO30U8liy3km5r2Fth3zQniTGpDvJ2fOw/BoWDrILj8rCSS2H/w3/CFzUazS3lt9u1pMTIZxIVpA4iE0kHLdXY9rHEic2ldTWClwBaRXChFQce1yNQSqfklhU2CR0ZRFHrODEGZSHLaFAv4POX80u5C4DnNv7Bz9LywJdc5X4jhv0Tg+dK/iE5c6IkQMNxbdECBEY4/BD8w4RbZAeTwz+ywzmsczZFsu8ugvLv42CpRnBc2nuxW22i4VbKe6n3NgimjhFz8l6hhU8rq6Y0IbOTkMJyMznzh3rL8+7CgxXrX4tm4bSyZW79twTZMYmHvLBTNpMYk1KU6V5qT2CZ25Fn/3vW7ld3ekakvyhRgsuuYI9kAIJDoxpRNdSP691rtM52JMvMy5axCFBBV6PdSuJNoHpDWx9eUsEFdFY0zfFjfrTDNqKxS560R8mY8V3E1C1QUx00HUj4XbBJwRJoLduoLwgMQ2KkFpZ+qjvko82uW4KLSSyqZUMGKrCzXJp3Q4ZfCMS9q7zTRVrDA0Gg+TmvJd8ykExM+IVHe+QhF/7c3jPWWxtfMXiESD5pcDJa2n+VSJivL5eeTJkSsl7Jz5OTMS4kdpygAPxDNkEd1uA65RzsVHKIsl4kJAibiIYkN6QSXIgYEDz6LKhlBbeyQPhFER4w/oOYGVxMPxKZe2ISRmk7O6J1SPdO+2sKMFVmXSZMkUsO7HFvgwz9poZ0ROTtS0UbRKYIlFTQnlqoz9hb0xlpy5oezdUpXx7OG644baH0rwVWLgNlJl0NMYIf7Xt5KSV3yXIDoNI8+VVQ+ef2+Zp4t8kT2T+DePr3+5fDNx7dDl6W7oTBnMRd361Ngz5Z1LzIRLyejmpZudsyXYNpiFbYzSUsF5bObEYDSfphV06iqlFCUNZmvj1+/PfoYfzw6fDNsqOuFOaZQJD3dUlhErTHVZmxlh6qTfGTirci7wGZFoy3MYZs8U9XDt55tonOwGCGhDuinPxEaY8eZE+rmhOHMCv7mkliHVBlRDSi208jyhN/AuFG3NuHKLp2ShBNEpxxn1prZNZIFZqvGFrfK8ws+UxA39X5MoxFVgGvT5bC8kkY1FSJbA2lLYq/eHI1G45aAey1jF8XuYIjJpJMYuiGWu41yQdAd/qhR6w3Q2xhV2xXCC3kmieKT+AAJ+OPRaDQ9+2ZF5guX3ToON/n3O+yY+qy4PtHEWoa2K/zy5ra8SnJlgyZ72Q7ehUDc+nn7PZewvJ9Ong7spT/SwsZlNPcAm3W7e4Rf1r2lp3zLXbpkoblR7rqfK2zVWZ4dQpahvjpYB/HO7JGvoaAhQKKysIadnsi2f5alqXFEaaVWJuDbM7rrFdiq1uf+dIAiQUawCP2CF6wEHUJySEIYe8Ezu2OS/nINu2WFgqIVvkWg66X1DC6Z51Svlb9/c+tgrT2cMpCTJ9Mz2G8+Fwa1rkot2DyeDhzoerCHjXqpGkxpz7Y0xdlbe+hDTaNodHufM7HNiw/f4HX+rOdxh3ObVvmbTJrbhJOWc6SpFHhv3t3YsODtZBt7WB1nY53lelmfy/RttghSbW+ZcTQ/pARiIUtRnwyGvhFwyPlwo1fT/pMLbmim8HrLXZV7+ZJ/Fy3+PQM+r0i+0oe0sBoBJ3glpDgJdh3UDsWjbe7NhbNndUJWNobt3q6omWBj1QNKHYM2dr0rKmWjMsvzLNggzPYRqKojuQSu6oBrZtQJyk6XwteaF6JOeKx1i2PKb+igbFofD1C6Y092G2dBZ7rdu0b+W6RZerTefWw35CPuob18OhTugp936Wbo7ocQkc2NcvgAZ8vVlVwVZBPP2sSMClWJ9c6A/Ct+FwuABp6DUXNdyt7el5puAhm+TGsPrjm2zy5smBj5H2EEDbBw++U7TAPpOnrLHo9Ntj+/vuhNt0SaUY23bV3PPGO5OIEDyPSYTDdG89cUBApguWOyvePv8R91B2ILWdatF73WXZcGePDnrSDyLa2L+Xhr66Tbeta/YuRfWXTe33KPOUCpYsiMsBeQjMoW0Q313faYeIMxbabeve8LgBHoiyiBWihZ8ccDkG1soFUw7p9EkPJeN+0cZlpcK1mIJOaSpth/iUc3HVPtCzdyf4OfnaNTAXsBIginYte/74DjAh6BdlLQ1ycUHup95JWf6fqSyakck5ai+QiFS7D4691ZgfRkHLLwsiE3Q2EfJhvVfx1dqnR5XtFZQhyMokfigaOVrnPoIOkXtjFjlkrzxfGuiEJ4be3Xu0KiFqpUjZK0mtQpS324Pia4e0bQvYLmSytmzFBwUJk43p77d0cwwS6pe6QMpHRApqMdXzMFnnVIyryAoBwgMBoKOa/Sz2x9DhAYq2xN0jyjqnfMt64ObHmOiPMa7tp739UsIpJn+rhJk+whAv89djtCV5biwFXLZRWcEC+h0m7H7J5CmXvvn937WTdB2IaIN3InGuptkfhvn/ntDgSkZYQh6izFTs5d45cne+TVU6kp7Eo63eK0JpLAZGxjnMBDbckGte2Tun1St9O2Neox9lsnrdKc9bRG9ta1uSI7yl+AT7rTDcLU1w2ueG24q1dDcQwTKNdGwkkiT1FZ34dZN7Xb2znUVxTce/pqIWPssr5xcmC7mxsoQbjhGRGIPWnqT18Q8Ws+lm6vzbLKX9WLObB/ODuzj7YYuznDY+5KmotmGTCMaye9cBQmOLYLosKtWwnFTtjYJJXLHHHowXgDG7kYr4QmNRXhdlopuhNuSuSyIOOAfupaH3/Mcw0Rckx1t+55u26nVJwv1u3lOvtxyPXQ+zagP7L9yOZx+5HNkw0D20qPHwVxatQRDk+2HkG6AXM8eT4UeHw6bA4Z3GwR1G6Ohwr6BBP6zz6OZ7mNa70sB0mroGiv5sOHMkv2x+5zpZORjU0L0ldSrtaJNeVo+wkG+d7x5NnoMTPG0TRlS++uN9BXDDf61vOD3qcLPLWtXeSVi+D4K8+Yp9gUIGgZ0fn8yH61sYWCOyE09223fNzRktTZLPsJSUzyDkny98zbrcfPeYv+iv8f4c+wTerrTeIRnU2iwTpnXSFBYmMx3oHd7uUO5E+xOJKSJ09ZVMaTVlZq7Dykg51HW7AN9kkfu+2PqQQGBlR0WncRSwKzKbueYRtbQojvDPfRDtMGs/7Mjv5a2/a4FsIrulB81f225GpyMh2K/fGZ+N59+PgV1s8qOZYQXE0am8fxAQGbAhp7meBqV+fpwLZPBdxXltLVSjYOpvmOgI/h+IhO0oXe/AKvro9YX5uNp33uE60p3S7/IrMfe8x+EsLiuuRvG9Mx7sn9mP40bBOtPh+iIr8MJrAeMCxhRGeG9OF3sJElNQaFv02OfbPSTzaKiBZeH1WT5ecA3jbz93YOnXgpRqz7DaOmrbERLgEhQwEtq4ds6rdnKei79dU6kz6/aQmNZ7FRjGdP4EpI3R7xcQ9enj19bhXw0egZRzDPx3+dWA2dPH/6rBuoODf2GAF6kOCHvHYS4mfCLw8Y4/f1L/6nkYn3UR5bTfc9xigaD7dbUN8z3pHX0adoqa5sxaL/UdymUb1JYDtvfqUfAL6907LyZ3O3m6znz57vFG62UsTS0eSxk+9nO+S6sYBP7iHaz+8RIzlGjynl6MVIXj7RxhgeQVxc4C87McOoTqQxuL39X18zfOE=")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
